# 宮崎空港 Normal Traffic Pattern Reference Paths

RJFM の AIP 転記データと航空大学校運用資料の場周経路条件から、RWY09 / RWY27 の NORTH / SOUTH pattern を生成します。これは風や機体運動を含まない幾何学的な `ReferencePath` です。

> ARP は参照・検算専用です。すべての pattern geometry は両 threshold の中点である **RWY Center Point** と True Bearing を基準に計算します。

## 1. AirportSpec と出力先

In [ ]:
from pathlib import Path
import math
import os

from IPython.display import display
from matplotlib import pyplot as plt

from sr22_course_simulator.data.airports import RJFM
from sr22_course_simulator.examples.miyazaki_traffic_patterns import (
    RJFM_COMBINED_PATTERN_FILENAME,
    RJFM_PATTERN_FILENAMES,
    build_rjfm_normal_patterns,
    rjfm_normal_pattern_specs,
    write_rjfm_normal_pattern_kmls,
)

default_artifact_root = (
    Path.cwd().parent / 'artifacts'
    if Path.cwd().name == 'notebooks'
    else Path.cwd() / 'artifacts'
)
artifact_root = Path(os.environ.get('SR22_ARTIFACT_DIR', default_artifact_root)).resolve()
output_dir = artifact_root / 'traffic-patterns'
output_dir.mkdir(parents=True, exist_ok=True)
display(RJFM)
print(f'KML output: {output_dir}')

## 2. RWY True Bearing と threshold

`threshold_a` は表示中の runway designation の着陸 threshold、`threshold_b` は reciprocal / departure-end threshold です。KML の方位計算には True Bearing だけを使います。

In [ ]:
runway_rows = [
    {
        'designation': runway.designation,
        'true_bearing_deg': runway.true_bearing_deg,
        'threshold_a': runway.threshold_a,
        'threshold_b': runway.threshold_b,
        'threshold_elevation_a_ft': runway.threshold_elevation_a_ft,
        'threshold_elevation_b_ft': runway.threshold_elevation_b_ft,
        'measured_length_m': runway.measured_length_m,
    }
    for runway in RJFM.runways
]
display(runway_rows)
print(f'MAG VAR at 2026.0 (reference only): {RJFM.variation_at(2026.0):.2f} deg')

## 3. RWY Center Point

`center_point = (threshold_a + threshold_b) / 2` を deterministic に計算します。ARP をこの計算へ渡しません。

In [ ]:
for runway in RJFM.runways:
    print(f'RWY{runway.designation} center: {runway.center_point}')
assert RJFM.runway('09').center_point == RJFM.runway('27').center_point

## 4. Pattern parameters

`CROSSWIND_EXTENSION_NM = 0.0` は明示的仮定です。値を変更しても departure end を基準に runway true-bearing 方向へ延長されます。

In [ ]:
PATTERN_ALTITUDE_FT = 1000.0
DOWNWIND_OFFSET_NM = 1.5
BASE_EXTENSION_NM = 1.2
CROSSWIND_EXTENSION_NM = 0.0
MAGNETIC_REFERENCE_YEAR = 2026.0

In [ ]:
pattern_specs = rjfm_normal_pattern_specs(
    altitude_ft=PATTERN_ALTITUDE_FT,
    downwind_offset_nm=DOWNWIND_OFFSET_NM,
    base_extension_nm=BASE_EXTENSION_NM,
    crosswind_extension_nm=CROSSWIND_EXTENSION_NM,
)
display(pattern_specs)
print(f'Magnetic reference at {MAGNETIC_REFERENCE_YEAR}: {RJFM.variation_at(MAGNETIC_REFERENCE_YEAR):.2f} deg (not used in geometry)')

## 5. 4 pattern を生成

各 path は `departure_reference → crosswind → downwind → base → final → threshold_return` の6点を持つ straight-segment polyline です。

In [ ]:
patterns = build_rjfm_normal_patterns(
    altitude_ft=PATTERN_ALTITUDE_FT,
    downwind_offset_nm=DOWNWIND_OFFSET_NM,
    base_extension_nm=BASE_EXTENSION_NM,
    crosswind_extension_nm=CROSSWIND_EXTENSION_NM,
)
for path in patterns:
    print(path.name, [point.label for point in path.points()])
assert len(patterns) == 4

## 6. 簡易可視化

In [ ]:
figure, axes = plt.subplots(figsize=(10, 8))
for path in patterns:
    longitudes = [point.position.longitude_deg for point in path.points()]
    latitudes = [point.position.latitude_deg for point in path.points()]
    axes.plot(longitudes, latitudes, marker='o', label=path.name)
center = RJFM.runway('09').center_point
axes.scatter([center.longitude_deg], [center.latitude_deg], marker='x', s=100, color='black', label='RWY Center Point')
axes.scatter([RJFM.reference_point.longitude_deg], [RJFM.reference_point.latitude_deg], marker='+', s=100, color='gray', label='ARP (reference only)')
axes.set_xlabel('Longitude [deg]')
axes.set_ylabel('Latitude [deg]')
axes.set_title('RJFM Normal Traffic Pattern Reference Paths')
axes.set_aspect(1.0 / math.cos(math.radians(center.latitude_deg)), adjustable='datalim')
axes.grid(True)
axes.legend()
plt.show()

## 7. KML 出力

4個の個別 KML と、4 Placemark を含む結合 KML を `artifacts/traffic-patterns/` に書き出します。

In [ ]:
written = write_rjfm_normal_pattern_kmls(
    output_dir,
    altitude_ft=PATTERN_ALTITUDE_FT,
    downwind_offset_nm=DOWNWIND_OFFSET_NM,
    base_extension_nm=BASE_EXTENSION_NM,
    crosswind_extension_nm=CROSSWIND_EXTENSION_NM,
)
expected_names = {*RJFM_PATTERN_FILENAMES, RJFM_COMBINED_PATTERN_FILENAME}
assert {path.name for path in written} == expected_names
assert all(path.is_file() for path in written)
for path in written:
    print(path)